> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 7 · Notebook 07 — Range, event and volatility option strategies

**Sessions:** S7 (Range-bound, either-way & volatility option strategies) · [Lesson plan](../../docs/lessons/PART_07_STRATEGY_LIBRARY.md) · graded labs in [`labs/part07/`](../../labs/part07/)

**You will:**
1. Read the market's expected move from the straddle.
2. Compare implied and realized moves over past events.
3. Separate a research-only volatility risk premium from a tradable signal.
4. Look at the worst month of a premium seller before its win rate.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic, built from regimes you know, and every strategy here is a **hypothesis** with a first-look evaluation: the honest backtest comes in Part 8.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p7lib.py is in notebooks/part07/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p7lib as p

p.use_course_style()

## 1. The implied move

The ATM straddle's price, as a fraction of spot, is the market's **expected absolute move** to expiry. A quick rule: `straddle ≈ √(2/π)·S·σ·√T ≈ 0.8·S·σ·√T`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def implied_move(straddle_price, spot):
    return straddle_price / spot

def straddle_rule(S, sigma, T):
    return float(np.sqrt(2 / np.pi) * S * sigma * np.sqrt(T))

S, sig, T = 600.0, 0.25, 7 / 365
exact = float(p.bsm_price(S, S, T, 0.0, 0.0, sig, 1) + p.bsm_price(S, S, T, 0.0, 0.0, sig, -1))
mine = [p.attempt(implied_move, exact, S), p.attempt(straddle_rule, S, sig, T)]
mine = p.check("implied move and rule of thumb", mine, [p.implied_move(exact, S), p.straddle_rule_of_thumb(S, sig, T)])
print(f"7-day ATM straddle ${exact:.2f} → implied move ±{mine[0]:.2%}; rule of thumb ${mine[1]:.2f}")

## 2. An event study

Forty past earnings events: the implied move from the straddle bought the day before, and the realized absolute move. Return `n`, the mean implied and realized moves, `edge = mean(realized − implied)` (a long straddle's rough edge per unit of spot) and the share of events where realized beat implied.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def event_study(implied, realized):
    implied, realized = np.asarray(implied, float), np.asarray(realized, float)
    return {"n": int(implied.size), "mean_implied": float(implied.mean()), "mean_realized": float(realized.mean()),
            "edge": float(np.mean(realized - implied)),
            "share_realized_above": float(np.mean(realized > implied))}

ev = p.earnings_events()
mine = p.attempt(event_study, ev.implied, ev.realized)
ref = {"n": 40, "mean_implied": float(ev.implied.mean()), "mean_realized": float(ev.realized.mean()),
       "edge": float((ev.realized - ev.implied).mean()), "share_realized_above": float((ev.realized > ev.implied).mean())}
mine = p.check("event_study", mine, ref)
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(ev.implied * 100, ev.realized * 100); ax.plot([0, 20], [0, 20], color="black", lw=0.8)
ax.set(xlabel="implied move, %", ylabel="realized move, %", title="Above the line: the straddle buyer won"); plt.show()
{k: round(v, 4) for k, v in mine.items()}

Here the straddle seller usually wins, and occasionally loses a lot (the fat tail). Forty events is a small sample: an edge estimate this noisy needs Part 8's statistics before it becomes a strategy.

## 3. The volatility risk premium: research vs trading

Implied vol is usually above the vol that then realizes: the **variance risk premium** that option sellers collect. Measuring it needs the **future** realized vol (`p.vrp_research`, fine for research). A trading signal can only use **trailing** realized vol, known at today's close. Write the tradable version: `iv − p.realized_vol(close, window)`.

In [ ]:
close, iv = p.vol_market()
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(iv.index, iv * 100, label="30-day implied"); ax.plot(close.index, p.realized_vol(close) * 100, label="21-day realized (trailing)")
ax.legend(); ax.set_title("Implied vol sits above realized, most of the time"); plt.show()

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def vrp_signal(iv_30d, close, window=21):
    return iv_30d - p.realized_vol(close, window)

mine = p.attempt(vrp_signal, iv, close)
mine = p.check("vrp_signal", mine, p.vrp_signal(iv, close))

cut = close.index[900]
research_full, research_cut = p.vrp_research(iv, close), p.vrp_research(iv[:cut], close[:cut])
signal_full, signal_cut = p.vrp_signal(iv, close), p.vrp_signal(iv[:cut], close[:cut])
same = lambda a, b: np.allclose(a.loc[:cut].dropna(), b.dropna().reindex(a.loc[:cut].dropna().index), equal_nan=True)
print("truncation test (values up to the cut unchanged when the future is removed):")
print("  research VRP:", same(research_full, research_cut), "  ← uses the future: research only")
print("  tradable VRP:", same(signal_full, signal_cut))

## 4. The premium seller's worst month

Sell a 30-day ATM straddle at the start of every month at the implied vol, hold to expiry. Look at the whole distribution, worst month first, before the win rate.

In [ ]:
S_, V_ = close.to_numpy(), iv.to_numpy()
pnl = []
for t in range(0, len(S_) - 21, 21):
    prem = p.bsm_price(S_[t], S_[t], 21 / 252, 0.0, 0.0, V_[t], 1) + p.bsm_price(S_[t], S_[t], 21 / 252, 0.0, 0.0, V_[t], -1)
    pnl.append((prem - abs(S_[t + 21] - S_[t])) / S_[t])
pnl = np.array(pnl)
fig, ax = plt.subplots()
ax.hist(pnl * 100, bins=30); ax.axvline(0, color="black", lw=0.8)
ax.set(xlabel="P&L per month, % of spot", title="Short ATM straddle, held to expiry"); plt.show()
print(f"win rate {np.mean(pnl > 0):.0%}; mean {pnl.mean():+.2%}; best month {pnl.max():+.2%}; worst month {pnl.min():+.2%}")

## Wrap-up

* The straddle prices the expected move; compare it with what events actually delivered.
* Research features may use the future; trading signals may not. Truncation tests keep them apart.
* Premium selling has a positive average and a brutal left tail: show the worst week first, size by it.
* Graded version: `labs/part07/week24_vol_hedging`.